# HTA-MAC paper-aligned B16 training (2026-08-06)

This notebook trains the **exploratory paper-aligned** environment: 100 static nodes, 100 m x 100 m field, central BS, 0.5 J initial energy, 20% exogenous balanced rotating CHs, trained solar HMM, thermal disabled, and B=16. HTA-MAC remains the only learned intervention.

**Claim boundary:** this is paper-aligned, not a reproduction of third-party results, and it does not replace the registered frozen HEART-CH experiment. Seeds 3100-3104 are prohibited here. The notebook calibrates on development seeds 2400-2404, trains three fresh lineages, audits them, evaluates the full network, and selects a candidate only if all five development trials meet the transferred QoS thresholds.

Expected NVIDIA L4 runtime: roughly 3-7 hours for all three lineages; actual time depends on Colab load. Results are checkpointed to Drive after every lineage.


In [ ]:
# Frozen user settings. Change only paths/download preference.
BUNDLE_PATH = '/content/HTA_MAC_PaperAligned_B16_Training_Bundle_20260806.zip'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/HTA_MAC_PaperAligned_B16_20260806'
SEEDS = [5299, 6299, 7299]
EPISODES = 500
DOWNLOAD_RESULTS_WHEN_COMPLETE = True
EXPECTED_BUNDLE_SHA256 = '919bf655617644be601c8beaaca38e11be319b5dfbe4c8b2d870541cadd7ee8a'
assert SEEDS == [5299, 6299, 7299]
assert EPISODES == 500


In [ ]:
# GPU, Drive, bundle discovery, checksum, safe extraction, and per-file manifest verification.
import csv, glob, hashlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU before training.')
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab; Drive mount skipped.')

bundle = Path(BUNDLE_PATH)
if not bundle.is_file():
    candidates = [
        Path(candidate)
        for candidate in glob.glob('/content/*PaperAligned*B16*Bundle*.zip')
    ]
    if len(candidates) != 1:
        raise FileNotFoundError(
            f'Expected one uploaded paper-aligned B16 bundle, found {candidates}'
        )
    bundle = candidates[0]

digest = hashlib.sha256(bundle.read_bytes()).hexdigest()
assert digest == EXPECTED_BUNDLE_SHA256, (digest, EXPECTED_BUNDLE_SHA256)

WORK = Path('/content/hta_mac_paper_aligned_b16')
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(bundle) as archive:
    for member in archive.infolist():
        normalized = member.filename.replace('\\', '/')
        member_path = PurePosixPath(normalized)
        if member_path.is_absolute() or '..' in member_path.parts:
            raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        target = WORK.joinpath(*member_path.parts)
        if member.is_dir() or normalized.endswith('/'):
            target.mkdir(parents=True, exist_ok=True)
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member, 'r') as source, target.open('wb') as destination:
            shutil.copyfileobj(source, destination)

stage2 = WORK / 'stage2'
repo = stage2 / 'hta-mac'
upstream = stage2 / 'final_repo'
manifest_path = stage2 / 'COLAB_PAPER_ALIGNED_B16_MANIFEST.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8-sig'))

OPTIMIZER_SEEDS = list(SEEDS)
DEVELOPMENT_SEEDS = [2400, 2401, 2402, 2403, 2404]
MAX_STEPS = 300
DRIVE_ROOT = Path(DRIVE_OUTPUT_DIR)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

assert manifest['optimizer_seeds'] == SEEDS
assert manifest['development_seeds'] == DEVELOPMENT_SEEDS
assert manifest['episodes'] == EPISODES and manifest['max_steps'] == MAX_STEPS
assert set(DEVELOPMENT_SEEDS).isdisjoint(
    manifest['prohibited_registered_held_out_seeds']
)
for entry in manifest['files']:
    target = stage2 / entry['path']
    assert target.is_file() and target.stat().st_size == entry['bytes'], entry['path']
    assert hashlib.sha256(target.read_bytes()).hexdigest() == entry['sha256'], entry['path']
print(
    'Verified bundle:', bundle.name,
    '| files:', len(manifest['files']),
    '| SHA256:', digest,
)


In [ ]:
# Install only runtime dependencies, then compile and run the complete validation suite.
dependencies = ['gymnasium', 'torch-geometric', 'scipy', 'pyyaml', 'pytest']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *dependencies], check=True)
subprocess.run([sys.executable, '-B', '-m', 'compileall', '-q', str(repo), str(upstream)], check=True)
test_command = [
    sys.executable, '-B', '-m', 'pytest', 'validation', '-q',
    '-p', 'no:cacheprovider',
]
test_result = subprocess.run(
    test_command, cwd=repo, text=True, capture_output=True,
)
print(test_result.stdout)
if test_result.stderr:
    print(test_result.stderr, file=sys.stderr)
if test_result.returncode != 0:
    raise RuntimeError(f'Validation failed with exit code {test_result.returncode}; see complete output above.')
print('Validation suite passed.')


In [ ]:
# Deterministic development-only return-scale calibration (never uses registered held-out seeds).
profile = repo / 'config/paper_aligned_hasani2025_b16.json'
qos = repo / 'config/paper_aligned_hasani2025_qos_constraints.json'
scale_drive = DRIVE_ROOT / 'paper_aligned_b16_return_scale.generated.json'
scale_local = repo / 'config/paper_aligned_b16_return_scale.generated.json'
calibration = [
    sys.executable, '-B', 'experiments/calibrate_paper_aligned_return_scale.py',
    '--environment-profile', str(profile), '--qos-constraint-config', str(qos),
    '--development-seeds', ','.join(map(str, DEVELOPMENT_SEEDS)),
    '--max-steps', str(MAX_STEPS), '--rollouts', '100', '--output', str(scale_drive),
]
subprocess.run(calibration, cwd=repo, check=True)
shutil.copy2(scale_drive, scale_local)
scale = json.loads(scale_local.read_text())
assert scale['status'] == 'frozen_development_scale' and not scale['held_out_seeds_used']
assert scale['development_seeds'] == DEVELOPMENT_SEEDS
print(json.dumps(scale, indent=2))


In [ ]:
# Train three independent fresh B16 lineages, audit each, and persist immediately to Drive.
def sync_tree(source, destination):
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(source, destination)

lineages = []
for seed in OPTIMIZER_SEEDS:
    run_name = f'paper_aligned_b16_equivariant_500ep_seed{seed}'
    local_run = repo / 'outputs/phase2' / run_name
    drive_run = DRIVE_ROOT / 'phase2' / run_name
    summary_path = drive_run / 'summary.json'
    audit_path = drive_run / 'foundation_audit.json'
    reusable = False
    if summary_path.exists() and audit_path.exists():
        old_summary = json.loads(summary_path.read_text())
        old_audit = json.loads(audit_path.read_text())
        reusable = old_summary.get('phase2_curriculum_gate_pass') is True and old_audit.get('status') == 'gate_pass'
    if reusable:
        print('Restoring completed lineage', seed)
        sync_tree(drive_run, local_run)
    else:
        if local_run.exists():
            shutil.rmtree(local_run)
        command = [
            sys.executable, '-B', 'experiments/train_phase2_dynamic_curriculum.py',
            '--episodes', str(EPISODES), '--max-steps', str(MAX_STEPS),
            '--development-seeds', ','.join(map(str, DEVELOPMENT_SEEDS)),
            '--optimizer-seed', str(seed), '--run-name', run_name,
            '--architecture', 'equivariant_set_branching', '--projection-budget', '16',
            '--reward-scale-config', str(scale_local), '--qos-constraint-config', str(qos),
            '--environment-profile', str(profile), '--normalize-input-blocks',
            '--learning-rate', '1e-5', '--trajectory-loss-weight', '1.0',
            '--concavity-loss-weight', '0.1', '--learn-every', '4',
            '--precision', 'fp32', '--stability-interval', '50', '--stability-tail-episodes', '100',
            '--device', 'cuda',
        ]
        log_dir = DRIVE_ROOT / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        log_path = log_dir / f'{run_name}.log'
        print('Starting lineage', seed, '| log:', log_path, flush=True)
        with log_path.open('w', encoding='utf-8') as log_handle:
            process = subprocess.Popen(
                command, cwd=repo, stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT, text=True, bufsize=1,
            )
            for output_line in process.stdout:
                print(output_line, end='', flush=True)
                log_handle.write(output_line)
                log_handle.flush()
            returncode = process.wait()
        if not local_run.exists():
            raise RuntimeError(
                f'Training startup failed for seed {seed} with exit code '
                f'{returncode}; inspect {log_path}'
            )
        sync_tree(local_run, drive_run)
        if returncode:
            raise RuntimeError(f'Training gate failed for seed {seed}; evidence saved at {drive_run}; log: {log_path}')
        checkpoint = local_run / 'branching_c51.pt'
        audit_local = local_run / 'foundation_audit.json'
        audit = [
            sys.executable, '-B', 'experiments/audit_phase2d_foundation.py', str(checkpoint),
            '--output', str(audit_local), '--max-steps', str(MAX_STEPS),
            '--development-seeds', ','.join(map(str, DEVELOPMENT_SEEDS)),
            '--environment-profile', str(profile), '--random-permutations', '20', '--targeted-swaps', '10',
        ]
        audit_result = subprocess.run(audit, cwd=repo)
        sync_tree(local_run, drive_run)
        if audit_result.returncode:
            raise RuntimeError(f'Foundation audit failed for seed {seed}; evidence saved at {drive_run}')
    summary = json.loads((local_run / 'summary.json').read_text())
    audit = json.loads((local_run / 'foundation_audit.json').read_text())
    assert summary['phase2_curriculum_gate_pass'] and audit['status'] == 'gate_pass'
    lineages.append({'seed': seed, 'run_name': run_name, 'checkpoint': str(local_run / 'branching_c51.pt')})
print('Structurally accepted lineages:', [x['seed'] for x in lineages])


In [ ]:
# Whole-network development evaluation against all bundled comparators.
for lineage in lineages:
    eval_name = lineage['run_name'] + '_dev_eval'
    local_eval = repo / 'outputs/phase3' / eval_name
    drive_eval = DRIVE_ROOT / 'phase3' / eval_name
    reusable = (drive_eval / 'summary.json').exists()
    if reusable:
        prior = json.loads((drive_eval / 'summary.json').read_text())
        reusable = prior.get('phase3_structural_gate_pass') is True and prior.get('seeds') == DEVELOPMENT_SEEDS
    if reusable:
        sync_tree(drive_eval, local_eval)
    else:
        if local_eval.exists():
            shutil.rmtree(local_eval)
        command = [
            sys.executable, '-B', 'experiments/run_phase3_pilot.py',
            '--seeds', ','.join(map(str, DEVELOPMENT_SEEDS)), '--horizon', str(MAX_STEPS),
            '--run-name', eval_name, '--skip-compatibility',
            '--hta-checkpoint', lineage['checkpoint'], '--hta-budget', '16',
            '--environment-profile', str(profile),
        ]
        result = subprocess.run(command, cwd=repo)
        assert local_eval.exists(), f'No evaluation output for {lineage["seed"]}'
        sync_tree(local_eval, drive_eval)
        if result.returncode:
            raise RuntimeError(f'Network evaluation failed for {lineage["seed"]}; evidence saved at {drive_eval}')
    lineage['evaluation'] = str(local_eval)
print('Completed network-wide evaluation for all lineages.')


In [ ]:
# Select only on development data using prespecified lexicographic QoS criteria.
qos_payload = json.loads(qos.read_text())
constraints = {
    'delivery_ratio_min': qos_payload['minimum_delivery_ratio'],
    'stale_drop_ratio_max': qos_payload['maximum_stale_drop_ratio'],
    'queue_fairness_min': qos_payload['minimum_queue_fairness'],
}
candidates = []
for lineage in lineages:
    raw_path = Path(lineage['evaluation']) / 'raw_trials.csv'
    with raw_path.open(newline='', encoding='utf-8') as handle:
        rows = [r for r in csv.DictReader(handle) if r['policy'] == 'hta_mac']
    assert len(rows) == len(DEVELOPMENT_SEEDS)
    trial_scores = []
    for row in rows:
        delivery = float(row['delivery_ratio'])
        stale = float(row['stale_drop_ratio'])
        fairness = float(row['queue_fairness'])
        passed = delivery >= constraints['delivery_ratio_min'] and stale <= constraints['stale_drop_ratio_max'] and fairness >= constraints['queue_fairness_min']
        violation = max(0, constraints['delivery_ratio_min']-delivery) + max(0, stale-constraints['stale_drop_ratio_max']) + max(0, constraints['queue_fairness_min']-fairness)
        trial_scores.append({'seed': int(row['seed']), 'pass': passed, 'violation': violation, 't_fnd': None if row['t_fnd'] in ('', 'None') else float(row['t_fnd']), 'throughput': float(row['throughput'])})
    import numpy as np
    candidate = {
        'optimizer_seed': lineage['seed'], 'run_name': lineage['run_name'],
        'joint_qos_pass_count': sum(x['pass'] for x in trial_scores),
        'median_constraint_violation': float(np.median([x['violation'] for x in trial_scores])),
        'median_t_fnd': float(np.median([x['t_fnd'] or 0.0 for x in trial_scores])),
        'median_throughput': float(np.median([x['throughput'] for x in trial_scores])),
        'trials': trial_scores,
    }
    candidates.append(candidate)
candidates.sort(key=lambda x: (-x['joint_qos_pass_count'], x['median_constraint_violation'], -x['median_t_fnd'], -x['median_throughput']))
best = candidates[0]
selection = {
    'status': 'development_candidate_selected' if best['joint_qos_pass_count'] == len(DEVELOPMENT_SEEDS) else 'no_candidate_qos_feasible',
    'claim_boundary': 'development_selection_only_not_confirmation_or_third_party_reproduction',
    'selected_optimizer_seed': best['optimizer_seed'] if best['joint_qos_pass_count'] == len(DEVELOPMENT_SEEDS) else None,
    'thresholds': constraints, 'ranking_rule': ['joint_qos_pass_count_desc', 'median_constraint_violation_asc', 'median_t_fnd_desc', 'median_throughput_desc'],
    'candidates': candidates, 'registered_held_out_seeds_used': False,
}
selection_path = DRIVE_ROOT / 'DEVELOPMENT_SELECTION.json'
selection_path.write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))


In [ ]:
# Package the complete evidence. A no-candidate outcome is preserved honestly rather than bypassed.
archive_base = Path('/content/HTA_MAC_PaperAligned_B16_Trained_Results_20260806')
final_archive = DRIVE_ROOT / (archive_base.name + '.zip')
sidecar = DRIVE_ROOT / (archive_base.name + '.zip.sha256')
for stale_artifact in (final_archive, sidecar):
    if stale_artifact.exists():
        stale_artifact.unlink()
archive = Path(shutil.make_archive(str(archive_base), 'zip', root_dir=DRIVE_ROOT))
shutil.copy2(archive, final_archive)
digest = hashlib.sha256(final_archive.read_bytes()).hexdigest()
sidecar.write_text(f'{digest}  {final_archive.name}\n')
print('RESULTS ZIP:', final_archive)
print('SHA256:', digest)
print('SELECTION STATUS:', selection['status'])
print('Reserved confirmation seeds 3400-3404 remain unused.')
if DOWNLOAD_RESULTS_WHEN_COMPLETE:
    try:
        from google.colab import files
        files.download(str(final_archive))
    except ImportError:
        print('Not running in Colab; automatic download skipped.')
